# 06 - Model Evaluation

**Project:** Student Placement & Salary Prediction  
**Goal of this notebook:** saved models ko test data par check karna, metrics compare karna aur best model chunna.

**Notebook order:** 01 Data Understanding -> 02 EDA -> 03 Data Preprocessing -> 04 Feature Engineering -> 05 Model Training -> 06 Model Evaluation. Ye notebook `notebook/` folder ke andar se run karo (paths `../datasets` aur `../models` relative hain).

## 1. Libraries aur saved cheezein load karna
Test data aur notebook 05 me save hue saare models load karte hain. (Extra Trees ~1.7 GB hai, load hone me thoda time aur RAM lagta hai.)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_test = pd.read_csv('../datasets/processed/X_test.csv')
y_test = pd.read_csv('../datasets/processed/y_test.csv').squeeze()
files = {'Linear Regression': 'linear_regression.pkl', 'Decision Tree': 'decision_tree.pkl', 'Random Forest': 'random_forest.pkl', 'Extra Trees': 'extra_trees.pkl', 'Gradient Boosting': 'gradient_boosting.pkl'}
models = {n: pickle.load(open('../models/' + f, 'rb')) for n, f in files.items()}
print('Loaded:', list(models))

## 2. Test set par predictions
Har model se unseen test data (10,892 students) par salary predict karwate hain.

In [ ]:
preds = {n: m.predict(X_test) for n, m in models.items()}
print({n: p[:3].round(2).tolist() for n, p in preds.items()})

## 3. Metrics nikalna
- **MAE:** average absolute galti (LPA me)
- **MSE / RMSE:** badi galtiyon ko zyada penalty
- **R2:** model data ka kitna variation samjha paaya (1 = perfect, 0 = sirf average)

In [ ]:
rows = []
for n, p in preds.items():
    mse = mean_squared_error(y_test, p)
    rows.append({'Model': n, 'MAE': mean_absolute_error(y_test, p), 'MSE': mse, 'RMSE': np.sqrt(mse), 'R2 Score': r2_score(y_test, p)})
metrics_df = pd.DataFrame(rows).sort_values('R2 Score', ascending=False).reset_index(drop=True)
metrics_df.to_csv('../models/model_metrics.csv', index=False)
metrics_df.round(4)

## 4. Metrics ka chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(metrics_df['Model'], metrics_df['R2 Score']); axes[0].set_title('R2 Score (higher is better)')
axes[1].bar(metrics_df['Model'], metrics_df['RMSE']); axes[1].set_title('RMSE in LPA (lower is better)')
for ax in axes:
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. Best model ka Actual vs Predicted aur Residuals
Best model (R2 ke hisaab se) ke liye: points jitne diagonal line ke paas, utna accha. Residual = actual - predicted.

In [ ]:
best = metrics_df.loc[0, 'Model']
p = preds[best]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
idx = np.random.RandomState(0).choice(len(y_test), 3000, replace=False)
axes[0].scatter(y_test.values[idx], p[idx], alpha=0.25, s=8)
lo, hi = y_test.min(), y_test.max()
axes[0].plot([lo, hi], [lo, hi], 'r--')
axes[0].set_xlabel('Actual salary'); axes[0].set_ylabel('Predicted salary'); axes[0].set_title('Actual vs Predicted: ' + best)
axes[1].hist(y_test.values - p, bins=40)
axes[1].set_title('Residuals: ' + best); axes[1].set_xlabel('Actual - Predicted')
plt.tight_layout()
plt.show()
print('Best model:', best)

## 6. Random Forest ki feature importance
Dashboard Random Forest use karta hai, isliye dekhte hain ki wo kin features par depend karta hai.

In [ ]:
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=X_test.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(fi.index, fi.values)
ax.set_title('Random Forest feature importance')
plt.tight_layout()
plt.show()
fi.sort_values(ascending=False).round(4).head(5)

## 7. Dashboard ke dikhaye numbers vs is notebook ke asli numbers
Dashboard ke `3_Model_Performance.py` page me 3 models ke metrics code me pehle se likhe hue (hard-coded) hain, jo kisi purane notebook se liye gaye the. Neeche unko is run ke asli test-set numbers se compare karte hain.

In [ ]:
dashboard_r2 = {'Linear Regression': 0.9797, 'Decision Tree': 0.9745, 'Random Forest': 0.9872}
cmp = pd.DataFrame({'Dashboard page (hard-coded) R2': dashboard_r2})
cmp['This notebook R2'] = metrics_df.set_index('Model')['R2 Score'].reindex(cmp.index).round(4)
cmp

## 8. Conclusion
CONCL